# Unit 08 - Peeking and Sequential Testing (Demo) · **V2 material**

**Atoms served:** `U08-A7` (**sequential testing / always-valid p-values** - **no video**; Johari et al. 2017 and this notebook carry it)

**Estimated runtime:** ~25 seconds

**After this notebook you can:** simulate how daily peeking inflates false positives in an A/A test, apply an always-valid boundary, and treat peeking as an engineering choice rather than a sin.

## Without code

1. Naive peeking: among 400 A/A simulations stopped at the first day with `p < 0.05`, roughly 25-40% stop "early" - far above 5%.
2. Always-valid boundary: the same simulations stop at or below the nominal 5% - in practice well below, because this boundary is conservative.
3. The lesson is not "never peek" but "peek with a method that preserves error rates."

**Note:** `V31` names p-hacking behaviour; this notebook makes the inflated false positive rate a number you produce.

## 1. The question

Your team checks an A/A experiment every day and stops the first time `p < 0.05`. There is zero true effect. How often do you "find" one anyway - and what changes with an always-valid boundary?

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

Each simulated experiment runs 14 daily looks on an A/A test (`n` users per day per arm, identical conversion rate).

We repeat the full monitoring loop hundreds of times so the false stop rate becomes a stable proportion.

In [ ]:
n_days = 14
n_per_arm_day = 200
p0 = 0.12
n_sims = 400
alpha = 0.05

## 4. The naive move

Peek daily with a fixed `p < 0.05` rule and stop at the first significant result.

Run the naive stopping rule across all simulations and count how often you stop "significant."

In [ ]:
def daily_pvalues(n_days, n_per_arm, p):
    ps = []
    cum_c, cum_t = 0, 0
    n_c, n_t = 0, 0
    for _ in range(n_days):
        y_c = np.random.binomial(1, p, n_per_arm)
        y_t = np.random.binomial(1, p, n_per_arm)
        cum_c += y_c.sum(); cum_t += y_t.sum()
        n_c += n_per_arm; n_t += n_per_arm
        _, pval = stats.ttest_ind_from_stats(cum_t/n_t, np.sqrt(cum_t/n_t*(1-cum_t/n_t)), n_t,
                                              cum_c/n_c, np.sqrt(cum_c/n_c*(1-cum_c/n_c)), n_c)
        ps.append(pval)
    return ps

naive_stops = 0
for _ in range(n_sims):
    ps = daily_pvalues(n_days, n_per_arm_day, p0)
    if any(p < alpha for p in ps):
        naive_stops += 1
naive_rate = naive_stops / n_sims
print('Naive peeking stop rate:', round(naive_rate, 3), '(nominal alpha=0.05)')

The stop rate is far above 5%. You did not discover an effect - you bought a false positive by looking too often.

## 5. What actually happens

**Always-valid boundary (mSPRT-style).** At each look, compare the z-statistic to a threshold that grows with sample size: `sqrt(2*log(1/alpha) + log(n))`.

Apply the same daily monitoring with the always-valid rule and count stops again.

In [ ]:
def msprt_boundary(n_total, alpha=0.05):
    return np.sqrt(2 * np.log(1 / alpha) + np.log(max(n_total, 2)))

def simulate_always_valid_stop(n_days, n_per_arm, p):
    cum_c, cum_t = 0, 0
    n_c, n_t = 0, 0
    for day in range(n_days):
        y_c = np.random.binomial(1, p, n_per_arm)
        y_t = np.random.binomial(1, p, n_per_arm)
        cum_c += y_c.sum(); cum_t += y_t.sum()
        n_c += n_per_arm; n_t += n_per_arm
        diff = cum_t/n_t - cum_c/n_c
        se = np.sqrt(cum_t/n_t*(1-cum_t/n_t)/n_t + cum_c/n_c*(1-cum_c/n_c)/n_c)
        z = diff / se if se > 0 else 0
        n_total = n_c + n_t
        if abs(z) > msprt_boundary(n_total, alpha):
            return True
    return False

av_stops = sum(simulate_always_valid_stop(n_days, n_per_arm_day, p0) for _ in range(n_sims))
av_rate = av_stops / n_sims
print('Always-valid stop rate:', round(av_rate, 3))

The stop rate drops back under the nominal 5% - in fact well under it. This simplified boundary is **conservative**: it protects the error rate at the cost of some power, which is why production implementations tune the mixture rather than using the closed form above. The lesson stands either way: peeking is fine when the boundary accounts for how often you looked.

Plot both stop rates side by side so the inflation and the fix are visible at a glance.

In [ ]:
fig, ax = plt.subplots()
ax.bar(['naive peeking', 'always-valid'], [naive_rate, av_rate])
ax.axhline(alpha, linestyle='--', label='nominal alpha')
ax.set_ylabel('fraction of A/A runs stopped early')
ax.set_title('Peeking inflates false positives; always-valid boundary restores alpha')
ax.legend()
plt.show()

Same data, two monitoring policies - one inflates error, one preserves it.

## 6. What you do about it

- Treat daily dashboard checks as **sequential looks**, not as a single final test (`U08-A7`).
- If you must peek, use **always-valid** methods (Johari et al., 2017) or platform tooling built on them.
- A fixed `p < 0.05` rule at every look is p-hacking even when nobody intended to cheat.

**When this matters less:** A pre-registered fixed horizon with no interim looks - then a plain test is honest.

---

**Takeaway:** Peeking is an engineering choice, not a moral failure - but only if the boundary you peek against was designed for repeated looks.

**Back to the unit:** [V2 unit 08](../V2/units/unit-08-power-duration-sample-size/README.md)